# PYNQ-Z2 ANC Project - Performance Analysis

This notebook will:
1. Analyze ANC convergence behavior in detail
2. Monitor step-size adaptation over extended periods
3. Measure convergence time and steady-state error
4. Evaluate filter weight adaptation
5. Perform stress testing with varying noise conditions

## Prerequisites
- Complete `03_anc_operation.ipynb` successfully
- ANC system operational with audio input
- Stable audio source connected

## 1. Import Libraries and Load Overlay

In [ ]:
from pynq import Overlay
from pynq import MMIO
import numpy as np
import matplotlib.pyplot as plt
import time

# Load the bitstream
overlay = Overlay('anc_project.bit')
print("Bitstream loaded successfully")

## 2. Access ANC Core Registers

In [ ]:
# Find ANC core IP
anc_core = None

for ip_name, ip_info in overlay.ip_dict.items():
    if 'anc' in ip_name.lower() or 'core' in ip_name.lower():
        anc_core = ip_name
        print(f"Found ANC core: {ip_name}")
        print(f"  Address: 0x{ip_info['phys_addr']:08X}")
        break

if anc_core:
    mmio = MMIO(overlay.ip_dict[anc_core]['phys_addr'], 
               overlay.ip_dict[anc_core]['addr_range'])
    print(f"MMIO object created for {anc_core}")
else:
    print("No ANC core found.")

## 3. Reset ANC System

Reset the system to start fresh convergence analysis.

In [ ]:
if anc_core:
    print("Resetting ANC system...")
    print("Press BTN0 on the board to reset.")
    print("Or use software reset if available.")
    
    # Wait for reset
    time.sleep(2)
    print("System reset complete.")
else:
    print("Skipping reset - no ANC core found.")

## 4. Long-Term Convergence Monitoring

Monitor ANC convergence over 30 seconds to observe full adaptation behavior.

In [ ]:
if anc_core:
    # Define register offsets
    MU_SHIFT_OFFSET = 0x00
    PRIMARY_IN_OFFSET = 0x04
    REF_NOISE_OFFSET = 0x08
    CLEANED_OUT_OFFSET = 0x0C
    DATA_READY_OFFSET = 0x10
    
    print("Starting long-term convergence monitoring (30 seconds)...")
    print("Ensure audio is playing during this period.")
    print("="*60)
    
    DURATION = 30  # seconds
    SAMPLE_RATE = 48828
    NUM_SAMPLES = int(DURATION * SAMPLE_RATE)
    
    error_buffer = []
    mu_shift_buffer = []
    timestamps = []
    
    start_time = time.time()
    sample_count = 0
    
    while sample_count < NUM_SAMPLES and time.time() - start_time < DURATION + 1:
        try:
            data_ready = mmio.read(DATA_READY_OFFSET)
            
            if data_ready & 1:
                primary = mmio.read(PRIMARY_IN_OFFSET)
                cleaned = mmio.read(CLEANED_OUT_OFFSET)
                mu_shift = mmio.read(MU_SHIFT_OFFSET)
                
                # Convert to signed
                if primary >= 32768:
                    primary = primary - 65536
                if cleaned >= 32768:
                    cleaned = cleaned - 65536
                
                error = cleaned  # e[n] is the cleaned output
                
                error_buffer.append(error)
                mu_shift_buffer.append(mu_shift)
                timestamps.append(time.time() - start_time)
                sample_count += 1
                
                if sample_count % 100000 == 0:
                    elapsed = time.time() - start_time
                    print(f"[{elapsed:.1f}s] Captured {sample_count} samples, mu_shift: {mu_shift}")
        except Exception as e:
            pass
        time.sleep(0.0001)
    
    error_buffer = np.array(error_buffer)
    mu_shift_buffer = np.array(mu_shift_buffer)
    timestamps = np.array(timestamps)
    
    print(f"\nCapture complete. Total samples: {len(error_buffer)}")
    print(f"Duration: {timestamps[-1]:.2f} seconds")
else:
    print("Skipping capture - no ANC core found.")

## 5. Plot Error Convergence

Visualize how the error (residual noise) decreases over time.

In [ ]:
if anc_core and len(error_buffer) > 0:
    # Calculate moving average for smoother visualization
    window_size = 1000
    error_ma = np.convolve(np.abs(error_buffer), np.ones(window_size)/window_size, mode='valid')
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # Raw error
    ax1.plot(timestamps[:len(error_buffer)], error_buffer, alpha=0.3, label='Raw Error')
    ax1.set_title('ANC Error Signal (Raw)')
    ax1.set_ylabel('Amplitude (16-bit)')
    ax1.legend()
    ax1.grid(True)
    
    # Moving average
    time_ma = timestamps[window_size-1:len(error_ma)+window_size-1]
    ax2.plot(time_ma, error_ma, color='red', linewidth=2, label='Moving Average (1000 samples)')
    ax2.set_title('ANC Error Convergence (Moving Average)')
    ax2.set_xlabel('Time (seconds)')
    ax2.set_ylabel('|Error|')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("No error data to plot.")

## 6. Plot Step-Size Adaptation

Visualize the step-size (mu_shift) changes over time.

In [ ]:
if anc_core and len(mu_shift_buffer) > 0:
    # Downsample for plotting
    downsample = 500
    mu_shift_plot = mu_shift_buffer[::downsample]
    time_plot = timestamps[::downsample]
    
    plt.figure(figsize=(14, 6))
    plt.plot(time_plot, mu_shift_plot, linewidth=2)
    plt.title('Step-Size Adaptation Over Time')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Step-Size Shift Value')
    plt.grid(True)
    
    # Add threshold lines
    for shift in range(1, 7):
        plt.axhline(y=shift, color='gray', linestyle=':', alpha=0.5)
    
    # Annotate regions
    plt.text(1, 1.5, 'Fast Adaptation', fontsize=10, color='red')
    plt.text(1, 3.5, 'Medium Adaptation', fontsize=10, color='orange')
    plt.text(1, 5.5, 'Fine Adaptation', fontsize=10, color='green')
    
    plt.show()
    
    print(f"Initial mu_shift: {mu_shift_buffer[0]}")
    print(f"Final mu_shift: {mu_shift_buffer[-1]}")
    print(f"Minimum mu_shift: {np.min(mu_shift_buffer)}")
    print(f"Maximum mu_shift: {np.max(mu_shift_buffer)}")
    print(f"Average mu_shift: {np.mean(mu_shift_buffer):.2f}")
else:
    print("No step-size data to plot.")

## 7. Calculate Convergence Metrics

In [ ]:
if anc_core and len(error_buffer) > 0:
    # Calculate error energy over time
    window_size = 5000
    error_energy = []
    
    for i in range(0, len(error_buffer) - window_size, window_size):
        segment = error_buffer[i:i+window_size]
        energy = np.mean(segment**2)
        error_energy.append(energy)
    
    error_energy = np.array(error_energy)
    
    # Find convergence point (when energy drops below 10% of initial)
    initial_energy = error_energy[0]
    threshold = 0.1 * initial_energy
    
    convergence_idx = np.where(error_energy < threshold)[0]
    if len(convergence_idx) > 0:
        convergence_sample = convergence_idx[0] * window_size
        convergence_time = convergence_sample / SAMPLE_RATE
    else:
        convergence_time = None
    
    # Steady-state error (last 5 seconds)
    steady_state_samples = int(5 * SAMPLE_RATE)
    steady_state_error = error_buffer[-steady_state_samples:]
    steady_state_rms = np.sqrt(np.mean(steady_state_error**2))
    
    print("Convergence Metrics:")
    print("="*50)
    print(f"Initial error energy: {initial_energy:.2f}")
    print(f"Final error energy: {error_energy[-1]:.2f}")
    print(f"Energy reduction: {(1 - error_energy[-1]/initial_energy)*100:.2f}%")
    print(f"Convergence time: {convergence_time:.2f} seconds" if convergence_time else "Convergence time: Not converged")
    print(f"Steady-state RMS error: {steady_state_rms:.2f}")
    
    # Plot error energy
    plt.figure(figsize=(12, 4))
    plt.semilogy(np.arange(len(error_energy)) * window_size / SAMPLE_RATE, error_energy)
    plt.axhline(y=threshold, color='r', linestyle='--', label='10% threshold')
    plt.title('Error Energy Over Time')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Error Energy (log scale)')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("No error data for metrics.")

## 8. Step-Size Distribution Analysis

Analyze how much time the system spends in each adaptation mode.

In [ ]:
if anc_core and len(mu_shift_buffer) > 0:
    # Count occurrences of each shift value
    shift_counts = np.bincount(mu_shift_buffer, minlength=7)
    shift_percentages = shift_counts / len(mu_shift_buffer) * 100
    
    print("Step-Size Distribution:")
    print("="*40)
    for shift in range(1, 7):
        print(f"Shift {shift}: {shift_percentages[shift]:.2f}% ({shift_counts[shift]} samples)")
    
    # Plot distribution
    plt.figure(figsize=(10, 6))
    bars = plt.bar(range(1, 7), shift_percentages[1:7], color=['red', 'orange', 'yellow', 'lightgreen', 'green', 'darkgreen'])
    plt.xlabel('Step-Size Shift Value')
    plt.ylabel('Percentage of Time (%)')
    plt.title('Step-Size Adaptation Distribution')
    plt.grid(True, axis='y')
    
    # Add value labels on bars
    for bar, pct in zip(bars, shift_percentages[1:7]):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                f'{pct:.1f}%', ha='center', va='bottom')
    
    plt.show()
else:
    print("No step-size data for distribution analysis.")

## 9. Adaptive Response to Changes

Test how the ANC responds to sudden changes in noise level.

In [ ]:
if anc_core:
    print("Testing adaptive response to noise changes...")
    print("="*50)
    print("Phase 1: Normal noise (0-10 seconds)")
    print("Phase 2: Increase noise (10-20 seconds)")
    print("Phase 3: Decrease noise (20-30 seconds)")
    print("\nPlease vary noise level manually during each phase.")
    
    PHASE_DURATION = 10  # seconds per phase
    error_by_phase = []
    mu_shift_by_phase = []
    
    for phase in range(3):
        print(f"\nStarting Phase {phase + 1}...")
        
        phase_error = []
        phase_mu = []
        start_time = time.time()
        
        while time.time() - start_time < PHASE_DURATION:
            try:
                data_ready = mmio.read(DATA_READY_OFFSET)
                if data_ready & 1:
                    cleaned = mmio.read(CLEANED_OUT_OFFSET)
                    mu_shift = mmio.read(MU_SHIFT_OFFSET)
                    
                    if cleaned >= 32768:
                        cleaned = cleaned - 65536
                    
                    phase_error.append(cleaned)
                    phase_mu.append(mu_shift)
            except:
                pass
            time.sleep(0.0001)
        
        error_by_phase.append(np.array(phase_error))
        mu_shift_by_phase.append(np.array(phase_mu))
        
        rms_error = np.sqrt(np.mean(np.array(phase_error)**2))
        avg_mu = np.mean(phase_mu)
        print(f"Phase {phase + 1} complete: RMS error = {rms_error:.2f}, Avg mu_shift = {avg_mu:.2f}")
    
    # Plot results
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
    
    phase_labels = ['Normal', 'High Noise', 'Low Noise']
    rms_errors = [np.sqrt(np.mean(e**2)) for e in error_by_phase]
    avg_mus = [np.mean(m) for m in mu_shift_by_phase]
    
    ax1.bar(phase_labels, rms_errors, color=['blue', 'red', 'green'])
    ax1.set_title('RMS Error by Phase')
    ax1.set_ylabel('RMS Error')
    ax1.grid(True, axis='y')
    
    ax2.bar(phase_labels, avg_mus, color=['blue', 'red', 'green'])
    ax2.set_title('Average Step-Size by Phase')
    ax2.set_ylabel('Average mu_shift')
    ax2.grid(True, axis='y')
    
    plt.tight_layout()
    plt.show()
else:
    print("Skipping adaptive response test - no ANC core found.")

## 10. Filter Weight Monitoring (if accessible)

If the design exposes filter weights, monitor their adaptation.

In [ ]:
if anc_core:
    # Check if filter weights are accessible
    # This depends on your design - adjust offsets accordingly
    WEIGHTS_OFFSET = 0x20  # Example offset
    NUM_TAPS = 16
    
    print("Attempting to read filter weights...")
    
    try:
        weights = []
        for i in range(NUM_TAPS):
            weight = mmio.read(WEIGHTS_OFFSET + i * 4)
            if weight >= 32768:
                weight = weight - 65536
            weights.append(weight)
        
        weights = np.array(weights)
        print(f"Filter weights: {weights}")
        
        plt.figure(figsize=(10, 4))
        plt.stem(range(NUM_TAPS), weights)
        plt.title('Current Filter Weights')
        plt.xlabel('Tap Index')
        plt.ylabel('Weight Value')
        plt.grid(True)
        plt.show()
    except Exception as e:
        print(f"Filter weights not accessible: {e}")
        print("(This is expected if weights are not exposed to AXI interface)")
else:
    print("Skipping weight monitoring - no ANC core found.")

## 11. Performance Summary Report

In [ ]:
if anc_core and len(error_buffer) > 0:
    print("="*60)
    print("ANC PERFORMANCE ANALYSIS REPORT")
    print("="*60)
    
    print("\n1. CONVERGENCE ANALYSIS:")
    print(f"   - Initial error energy: {initial_energy:.2f}")
    print(f"   - Final error energy: {error_energy[-1]:.2f}")
    print(f"   - Energy reduction: {(1 - error_energy[-1]/initial_energy)*100:.2f}%")
    print(f"   - Convergence time: {convergence_time:.2f} seconds" if convergence_time else "   - Convergence time: Not converged")
    
    print("\n2. STEADY-STATE PERFORMANCE:")
    print(f"   - Steady-state RMS error: {steady_state_rms:.2f}")
    print(f"   - Final step-size shift: {mu_shift_buffer[-1]}")
    
    print("\n3. STEP-SIZE ADAPTATION:")
    print(f"   - Average mu_shift: {np.mean(mu_shift_buffer):.2f}")
    print(f"   - Range: [{np.min(mu_shift_buffer)}, {np.max(mu_shift_buffer)}]")
    
    print("\n4. ADAPTIVE RESPONSE:")
    if len(rms_errors) == 3:
        print(f"   - Normal noise RMS: {rms_errors[0]:.2f}")
        print(f"   - High noise RMS: {rms_errors[1]:.2f}")
        print(f"   - Low noise RMS: {rms_errors[2]:.2f}")
    
    print("\n" + "="*60)
    print("Analysis complete.")
else:
    print("No data for performance report.")

## 12. Summary

### Analysis Completed:
- [ ] Long-term convergence monitoring
- [ ] Error convergence visualization
- [ ] Step-size adaptation tracking
- [ ] Convergence metrics calculation
- [ ] Step-size distribution analysis
- [ ] Adaptive response testing
- [ ] Filter weight monitoring (if accessible)

### Key Findings:
- Convergence time: (See cell 7)
- Steady-state error: (See cell 7)
- Step-size adaptation: (See cell 8)

### Next Steps:
- Proceed to `05_bypass_mode.ipynb` to verify bypass functionality
- Adjust step-size controller parameters if convergence is too slow
- Test with different noise types (white, colored, impulsive)